# Type Hints & Static Type Checking

**Topic**: Type Annotations, Type Checking, and Type Safety  
**Goal**: Master Python type hints for better code quality and IDE support

## What You'll Learn
- What type hints are and why they matter
- Basic and advanced type annotations
- Using the `typing` module
- Generic types and TypeVar
- Type aliases and NewType
- Protocol for structural typing
- TypedDict for typed dictionaries
- Static type checking with mypy
- Real-world patterns


## Setup

Type hints are built into Python 3.5+. We'll use Python 3.9+ syntax (built-in types).


In [6]:
# Import typing utilities
from typing import (
    Optional, Union, Any, Literal, Callable,
    TypeVar, Generic, Protocol, TypeAlias
)
from typing_extensions import TypedDict  # For older Python versions
import sys

print(f"Python version: {sys.version}")
print("Type hints are built-in, no installation needed!")


Python version: 3.11.8 (main, Nov 23 2025, 14:33:42) [Clang 17.0.0 (clang-1700.0.13.5)]
Type hints are built-in, no installation needed!


## 1. Why Type Hints Matter

### The Problem: Silent Type Bugs

Without type hints, type errors only appear at runtime.

In [4]:
# Problem: No type hints = bugs caught at runtime
def calculate_discount(price, discount):
    """Calculate price after discount"""
    return price - (price * discount)

# Intended usage (discount as decimal)
result1 = calculate_discount(100, 0.2)
print(f"Correct: ${result1}")   # $80.0

# Accidental bug (discount as percentage)
result2 = calculate_discount(100, 20)
print(f"Bug: ${result2}")   #-1900.0 - Wrong

print("\nBug only discovered at runtime")

Correct: $80.0
Bug: $-1900

Bug only discovered at runtime


### Solution: Type hints catch bugs early

In [6]:
def calculate_discount_typed(price: float, discount: float) -> float:
    """Calculate price after discount

    Args:
        price: Original price
        discount: Discount as decimal (e.g. 0.2 for 20%)

    Returns:
        Price after discount
    """
    return price - (price * discount)

# Correct usage
result1 = calculate_discount_typed(100.0, 0.2)
print(f"Correct: ${result1}")

# Wrong usage - mypy would catch this before running!
# result2 = calculate_discount_typed(100, 20)  # Type error!

print("\nWith type hints, mypy catches bugs before running!")
print("Run: mypy your_script.py")


Correct: $80.0

With type hints, mypy catches bugs before running!
Run: mypy your_script.py


## 2. Basic Type Annotations

### Variable Annotations

In [1]:
# basic type annotations
name: str = "John Doe"
age: int = 30
price: float = 10.99
is_active: bool = True

print(f"name: {name} (type: {type(name).__name__})")
print(f"age: {age} (type: {type(age).__name__})")
print(f"price: {price} (type: {type(price).__name__})")
print(f"is_active: {is_active} (type: {type(is_active).__name__})")

# type declaration without initial value
username: str
count: int

# later assignment
username = "john_doe"
count = 42

print(f"\nDeferred assignment:")
print(f"username: {username}")
print(f"count: {count}")


name: John Doe (type: str)
age: 30 (type: int)
price: 10.99 (type: float)
is_active: True (type: bool)

Deferred assignment:
username: john_doe
count: 42


### Function Annotations

In [2]:
# function with type hints
def greet(name: str) -> str:
    """Greet a person by name"""
    return f"Hello, {name}"

def add(a: int, b: int) -> int:
    """Add two integers"""
    return a+ b

def divide(a: float, b: float) -> float:
    """Divide two floats"""
    if b == 0:
        raise ValueError("Cannot divide by zero")
    return a / b

# Test the functions
print(greet("Alice"))
print(f"5 + 3 = {add(5, 3)}")
print(f"10.0 / 2.0 = {divide(10.0, 2.0)}")

# Function annotations don't prevent wrong types at runtime
# They're for static analysis tools like mypy
result = add(5.5, 3.2)  # Python allows this (returns 8.7)
print(f"\nPython allows: add(5.5, 3.2) = {result}")
print("But mypy would flag this as a type error!")

Hello, Alice
5 + 3 = 8
10.0 / 2.0 = 5.0

Python allows: add(5.5, 3.2) = 8.7
But mypy would flag this as a type error!


## 3. Collection Types

Python 3.9+ uses lowercase built-in types directly.

In [3]:
# collection type annotations (Python 3.9+)
names: list[str] = ["Alice", "Bob", "Charlie"]
ages: dict[str, int] = {"Alice" : 30, "Bob": 25}
unique_ids: set[int] = {1, 2, 3, 4, 5}
coordinates: tuple[float, float] = (10.5, 20.3)

print("List of strings:", names)
print("Dict of str->int:", ages)
print("Set of ints:", unique_ids)
print("Tuple of floats:", coordinates)

# nested collections
matrix: list[list[int]] = [
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 9]
]

student_grades: dict[str, list[float]] = {
    "Alice": [95.0, 87.5, 92.0],
    "Bob": [88.0, 91.5, 85.0]
}

print("\nNested list:", matrix[0])
print("Nested dict:", student_grades["Alice"])

List of strings: ['Alice', 'Bob', 'Charlie']
Dict of str->int: {'Alice': 30, 'Bob': 25}
Set of ints: {1, 2, 3, 4, 5}
Tuple of floats: (10.5, 20.3)

Nested list: [1, 2, 3]
Nested dict: [95.0, 87.5, 92.0]


## 4. Optional Types

`Optional[Type]` means the value can be `Type` or `None`.

In [7]:
# optional types
def find_user(user_id: int) -> Optional[str]:
    """Find user by ID, return None if not found"""
    users = {1: "Alice", 2: "Bob"}
    return users.get(user_id)

# Test with existing and non-existing users
user1 = find_user(1)
user2 = find_user(999)

print(f"User 1: {user1}")  # Alice
print(f"User 999: {user2}")  # None

# type checker knows user might be None
if user1 is not None:
    print(f"Found: {user1.upper()}")  # Safe to call .upper()
else:
    print("User not found")

# modern syntax (Python 3.10+)
def find_user_modern(user_id: int) -> str | None:
    """Same as OPtional[str]"""
    users = {1: "Alice", 2: "Bob"}
    return users.get(user_id)

print(f"\nModern syntax: {find_user_modern(2)}")

User 1: Alice
User 999: None
Found: ALICE

Modern syntax: Bob


## 5. Union Types

`Union[Type1, Type2]` means value can be one of multiple types.

In [8]:
# Union types
def process_id(id_value: Union[int, str]) -> str:
    """process ID that can be int or str"""
    if isinstance(id_value, int):
        return f"id-{id_value:06d}"
    else:
        return id_value.upper()

# Test with different types
print(process_id(123))      # "ID-000123"
print(process_id("abc"))    # "ABC"

id-000123
ABC


In [10]:
# Modern syntax (Python 3.10+)
def process_value(val: int | float | str) -> str:
    """Accept multiple types"""
    return str(val)

print(f"\nMultiple types: {process_value(42)}")
print(f"Multiple types: {process_value(3.14)}")
print(f"Multiple types: {process_value('hello')}")



Multiple types: 42
Multiple types: 3.14
Multiple types: hello


## 6. Literal Types

`Literal` restricts values to specific literals.

In [11]:
# Literal types
def set_log_level(level: Literal["DEBUG", "INFO", "WARNING", "ERROR"]) -> None:
    """Set logging level - only accepts specific strings"""
    print(f"Log level set to : {level}")

# Valid calls
set_log_level("DEBUG")
set_log_level("INFO")
set_log_level("ERROR")

Log level set to : DEBUG
Log level set to : INFO
Log level set to : ERROR


In [13]:
# Invalid call - would be caught by mypy
set_log_level("TRACE")  # Error: not a valid literal
set_log_level("debug")  # Error: case matters

Log level set to : TRACE
Log level set to : debug


In [15]:
# Literal with multiple types
def process_mode(mode: Literal[1, 2, "auto", "manual"]) -> None:
    """Accept specific int or string values"""
    print(f"Mode: {mode}")

process_mode(1)
process_mode("auto")
# process_mode(3)  # Error: not a valid literal

Mode: 1
Mode: auto


## 7. Callable Types

`Callable` annotates functions passed as arguments.

In [16]:
# callable types
def apply_operation(
        x: int,
        y: int,
        operation: Callable[[int, int], int]
) -> int:
    """Apply an operation to two numbers"""
    return operation(x, y)

# define operations
def add_nums(a:int, b: int) -> int:
    return a+b

def multiply_nums(a: int, b: int) -> int:
    return a*b

# use them
result1 = apply_operation(5, 3, add_nums)
result2 = apply_operation(5, 3, multiply_nums)

print(f"5 + 3 = {result1}")
print(f"5 * 3 = {result2}")

# Lambda functions work too
result3 = apply_operation(10, 2, lambda a, b: a - b)
print(f"10 - 2 = {result3}")

5 + 3 = 8
5 * 3 = 15
10 - 2 = 8


## 8. Generic Types with TypeVar

Create reusable, type-safe code that works with any type.

In [17]:
# Generic function with TypeVar
T = TypeVar('T')

def get_first(items: list[T]) -> T:
    """Get first item from list - works with any type"""
    if not items:
        raise ValueError("List is empty")
    return items[0]

# Type is inferred from usage
first_num = get_first([1, 2, 3])
first_str = get_first(["a", "b", "c"])
first_float = get_first([1.5, 2.5, 3.5])

print(f"First number: {first_num} (type: {type(first_num).__name__})")
print(f"First string: {first_str} (type: {type(first_str).__name__})")
print(f"First float: {first_float} (type: {type(first_float).__name__})")

First number: 1 (type: int)
First string: a (type: str)
First float: 1.5 (type: float)


In [18]:
# Generic function preserves type
def swap(a: T, b: T) -> tuple[T, T]:
    """Swap two values of the same type"""
    return b, a

x, y = swap(1, 2)
print(f"\nSwap integers: {x}, {y}")

a, b = swap("hello", "world")
print(f"Swap strings: {a}, {b}")


Swap integers: 2, 1
Swap strings: world, hello


## 9. Generic Classes

Create classes that work with any type while maintaining type safety

In [22]:
# Generic class
T = TypeVar('T')

class Stack(Generic[T]):
    """Generic stack that works with any type"""

    def __init__(self) -> None:
        self._items: list[T] = []

    def push(self, item: T) -> None:
        """Add item to stack"""
        self._items.append(item)

    def pop(self) -> T:
        """Remove and return top item"""
        if not self._items:
            raise IndexError("Stack is empty")
        return self._items.pop()

    def peek(self) -> T:
        """Return top item without removing"""
        if not self._items:
            raise IndexError("Stack is empty")
        return self._items[-1]

    def is_empty(self) -> bool:
        """Check if stack is empty"""
        return len(self._items) == 0

    def __len__(self) -> int:
        return len(self._items)

# Type-safe integer stack
int_stack: Stack[int] = Stack()
int_stack.push(1)
int_stack.push(2)
int_stack.push(3)

print("Integer stack:")
print(f"  Pop: {int_stack.pop()}")  # 3
print(f"  Peek: {int_stack.peek()}")  # 2
print(f"  Size: {len(int_stack)}")  # 2

# Type-safe string stack
str_stack: Stack[str] = Stack()
str_stack.push("hello")
str_stack.push("world")

print("\nString stack:")
print(f"  Pop: {str_stack.pop()}")  # world
print(f"  Peek: {str_stack.peek()}")  # hello

# Type error - mypy would catch this
# int_stack.push("string")  # Error: expected int, got str

Integer stack:
  Pop: 3
  Peek: 2
  Size: 2

String stack:
  Pop: world
  Peek: hello


## 10. Type Aliases

Create readable names for complex types.

In [23]:
# Type aliases make code more readable
UserId: TypeAlias = int
Username: TypeAlias = str
Point: TypeAlias = tuple[float, float]
UserData: TypeAlias = dict[str, int | str | list[str]]

def get_user(user_id: UserId) -> Username:
    """Get username by ID"""
    users = {1: "Alice", 2: "Bob"}
    return users.get(user_id, "Unknown")

def calculate_distance(p1: Point, p2: Point) -> float:
    """Calculate distance between two points"""
    import math
    return math.sqrt((p2[0] - p1[0])**2 + (p2[1] - p1[1])**2)

# Use the aliases
username = get_user(1)
print(f"User: {username}")

point1: Point = (0.0, 0.0)
point2: Point = (3.0, 4.0)
distance = calculate_distance(point1, point2)
print(f"Distance: {distance}")

# Complex type alias
user: UserData = {
    "id": 123,
    "name": "Alice",
    "roles": ["admin", "user"]
}
print(f"\nUser data: {user}")

User: Alice
Distance: 5.0

User data: {'id': 123, 'name': 'Alice', 'roles': ['admin', 'user']}


## 11. NewType - Distinct Types

Create distinct types for type checking (same at runtime).